In [ ]:
import numpy as np
import pandas as pd
import os

# 指定目录路径
# Run this notebook with the working directory set to the data_preprocessing folder.
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
folder_path = os.environ.get("CGLR_RAW_DATA_DIR", os.path.join(project_root, "data", "raw"))


# 文件名称按顺序排列
file_names = ['TRD_Dalyr.csv', 'TRD_Dalyr1.csv', 'TRD_Dalyr2.csv', 'TRD_Dalyr3.csv', 'TRD_Dalyr4.csv','TRD_Dalyr5.csv','TRD_Dalyr6.csv','TRD_Dalyr7.csv','TRD_Dalyr8.csv','TRD_Dalyr9.csv']

# 初始化一个空的列表来存储每个数据集
dfs = []

# 循环读取每个文件并存入列表
for file in file_names:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)  # 读取CSV文件
    dfs.append(df)  # 将DataFrame存入列表

# 按顺序拼接所有 DataFrame
df_combined = pd.concat(dfs, ignore_index=True)

# 显示拼接后的结果
print(df_combined)

In [ ]:
filtered_df0 = df_combined

In [ ]:
# 将 Trddt 列转换为 datetime 格式
filtered_df0['Trddt'] = pd.to_datetime(filtered_df0['Trddt'], errors='coerce')

# 定义日期范围
start_date = '2016-05-31'
end_date = '2023-06-01'

# 筛选出日期范围内的数据
filtered_by_date = filtered_df0

# 显示筛选后的数据
filtered_by_date

# 日个股交易股数/(日个股流通市值/日个股收盘价) Dnshrtrd，Dsmvosd，Clsprc¶

In [ ]:
# 计算 Dnshrtrd / Dsmvosd
filtered_by_date['intermediate_value'] = filtered_by_date['Dnshrtrd'] / filtered_by_date['Dsmvosd']

# 用中间值除以 Clsprc，得到换手率
filtered_by_date['换手率'] = filtered_by_date['intermediate_value'] / filtered_by_date['Clsprc']

# 删除不需要的中间列
filtered_by_date = filtered_by_date.drop(columns=['intermediate_value'])

# 显示结果
filtered_by_date[['Dnshrtrd', 'Dsmvosd', 'Clsprc', '换手率']]

# Opnprc 日开盘价 Hiprc 日最高价 Loprc 日最低价 Clsprc 日收盘价 ChangeRatio 涨跌幅

In [ ]:
# 使用 rename() 修改列名
filtered_by_date = filtered_by_date.rename(columns={
    'Opnprc': '日开盘价',
    'Hiprc': '日最高价',
    'Loprc': '日最低价',
    'Clsprc': '日收盘价',
    'ChangeRatio': '涨跌幅'
})

# 显示修改后的 DataFrame
print(filtered_by_date.head())

# 只要上证A和深证A

In [ ]:
filtered_data = filtered_by_date[(filtered_by_date['Markettype'] == 1) | (filtered_by_date['Markettype'] == 4)]

In [ ]:
len(filtered_data['Stkcd'].unique())

In [ ]:
selected_columns = filtered_data[['Stkcd', 'Trddt', '日开盘价', '日最高价', '日最低价', '日收盘价','换手率','涨跌幅','Dsmvtll']]

rolling(window=5).mean()：使用滚动窗口计算涨跌幅的移动平均值。window 参数定义了滚动窗口的大小，例如 5 日、10 日、20 日和 30 日。

In [ ]:
import pandas as pd

# 假设 selected_columns 是包含 Stkcd 和 涨跌幅 的 DataFrame

# 定义函数来计算每只股票的移动平均线
def calculate_moving_averages(df):
    df['5日均线'] = df['涨跌幅'].rolling(window=5).mean().fillna(0)
    df['10日均线'] = df['涨跌幅'].rolling(window=10).mean().fillna(0)
    df['20日均线'] = df['涨跌幅'].rolling(window=20).mean().fillna(0)
    df['30日均线'] = df['涨跌幅'].rolling(window=30).mean().fillna(0)
    return df

# 按 Stkcd 分组并应用移动平均线计算
selected_columns_with_ma = selected_columns.groupby('Stkcd').apply(calculate_moving_averages)

# 显示结果
selected_columns_with_ma

In [ ]:
selected_columns_with_ma[['开盘价变化率', '最高价变化率', '最低价变化率', '收盘价变化率']] = selected_columns_with_ma.groupby('Stkcd')[
    ['日开盘价', '日最高价', '日最低价', '日收盘价']
].pct_change()

In [ ]:
# 去除第四列“日收盘价”
selected_columns_with_ma = selected_columns_with_ma.drop(columns=['日收盘价','日最高价','日最低价','日收盘价'])

In [ ]:
selected_columns_with_ma

In [ ]:
# 定义日期范围
start_date = '2021-01-01'
end_date = '2022-12-31'

# 筛选出日期范围内的数据
selected_columns_with_ma2 = selected_columns_with_ma[(selected_columns_with_ma['Trddt'] >= start_date) & (selected_columns_with_ma['Trddt'] <= end_date)]

In [ ]:
filtered_data=selected_columns_with_ma2

# Dsmvtll [日个股总市值]
对于每一个股票stkcd，有很多天的数据，得到一张新表，新表记录每个股票在各自交易日中的平均日个股总市值，然后构成的新表格第一列为股票id，第二列为平均日个股总市值，并按平均日个股总市值从大到小

In [ ]:
import pandas as pd

# 计算每个股票的平均日开盘价
avg_Dsmvtll_by_stock = filtered_data.groupby('Stkcd')['Dsmvtll'].mean().reset_index()

# 重命名列名
avg_Dsmvtll_by_stock.columns = ['Stkcd', 'AvgDsmvtll']

# 按平均开盘价从大到小排序
avg_Dsmvtll_by_stock = avg_Dsmvtll_by_stock.sort_values(by='AvgDsmvtll', ascending=False)

# 输出新的表格
avg_Dsmvtll_by_stock

In [ ]:
big_stkcd=avg_Dsmvtll_by_stock.iloc[:900,].Stkcd
big_stkcd

# 查看股票的天数

In [ ]:
# 按 'Stkcd' 分组，并计算每个分组的条数
stock_counts = filtered_data.groupby('Stkcd').size().reset_index(name='Count')

# 显示每只股票的条数
stock_counts

In [ ]:
# 筛选出 Count 大于或等于 470 的股票
stocks_with_large_count = stock_counts[stock_counts['Count'] >= 485]

# 显示符合条件的股票数量
print(f"有 {len(stocks_with_large_count)} 只股票的 Count 大于或等于 972")

# 显示符合条件的股票列表
stocks_with_large_count

In [ ]:
new_stocks=stocks_with_large_count['Stkcd']
new_stocks

# 取大股和没有缺失值的交集

In [ ]:
# 如果 big_stkcd 和 new_stocks 是列表，先转换为集合
intersection = list(set(big_stkcd).intersection(set(new_stocks)))
len(intersection)

In [ ]:

# 筛选出仅在 stock_list 中的股票数据
filtered_df0 = filtered_data[filtered_data['Stkcd'].isin(intersection)]

# 显示筛选后的结果
filtered_df0

In [ ]:
len(filtered_df0['Stkcd'].unique())

In [ ]:
len(filtered_df0['Trddt'].unique())

In [ ]:
selected_columns_with_ma2 = filtered_df0

In [ ]:
import pandas as pd

# 假设 selected_columns 是您的 DataFrame

# 创建一个新的列 '涨跌幅分类'，如果 '涨跌幅' > 0 则为 1，否则为 0
selected_columns_with_ma2['涨跌幅分类'] = selected_columns_with_ma2['涨跌幅'].apply(lambda x: 1 if x > 0 else 0)

# 计算正和负的数量
positive_count = selected_columns_with_ma2['涨跌幅分类'].sum()  # 涨跌幅为 1 的数量
negative_count = len(selected_columns_with_ma2) - positive_count  # 涨跌幅为 0 的数量

# 计算正负数量的比例
ratio = positive_count / negative_count if negative_count != 0 else 'Inf'

# 输出结果
print(f"涨跌幅为正的数量: {positive_count}")
print(f"涨跌幅为负的数量: {negative_count}")
print(f"正负数量比例: {ratio}")

# 如果需要保存处理

In [ ]:
selected_columns_with_ma2

In [ ]:
selected_columns = selected_columns_with_ma2[['开盘价变化率', '最高价变化率', '最低价变化率', '收盘价变化率', '换手率','5日均线','10日均线','20日均线','30日均线', '涨跌幅']]

In [ ]:
selected_columns

In [ ]:
# 首先重置索引
selected_columns_reset = selected_columns.reset_index(drop=True)


# 保存路径
data_path = os.environ.get("CGLR_OUTPUT_DATA_PATH", os.path.join(project_root, "CGLR_main", "data", "CAS_S.csv"))
output_dir = os.path.dirname(data_path)
if output_dir:
    os.makedirs(output_dir, exist_ok=True)


# 将前9列保存为 CSV
selected_columns_reset.to_csv(data_path, index=False, header=False)



# 返回成功消息
data_path

# 772家公司 485天交易日